In [3]:
from pymongo import MongoClient
from random import randint
from pprint import pprint

import warnings
warnings.filterwarnings('ignore')

# Create a connection

In [4]:
host = "mongodb://mongo:27017/", #if within docker.

# NOTE: if you are running this notebook in docker you need to 
# refer to the container name "mongodb://mongo:27017/"
# mongo = "mongo"

# we use the MongoClient to communicate with the running database instance.
myclient = MongoClient(
                    host,  
                    username='admin',
                    password='admin') #Mongo URI format

# Or you can use the attribute style 
# mydb = myclient.customer_db

# Create a Dtabase

In [6]:
mydb = myclient["customer_db"]

In [7]:
myclient.list_database_names()

['admin', 'config', 'local', 'mydb']

# Create a collection

In [8]:
customers = mydb['customers']

In [10]:
mydb.list_collection_names()

[]

In [11]:
myclient.list_database_names()

['admin', 'config', 'local', 'mydb']

## MongoDB is Lazy

In [12]:
first_customer_doc = {"name": "Jane", "age": 25, "gender": "female"}

In [ ]:
customers.insert_one(first_customer_doc)

In [14]:
mydb.list_collection_names()

['customers']

In [15]:
myclient.list_database_names()

['admin', 'config', 'customer_db', 'local', 'mydb']

In [17]:
c = customers.find_one()

In [21]:
c

{'_id': ObjectId('6926c236298f27673a93246f'),
 'name': 'Jane',
 'age': 25,
 'gender': 'female'}

### About IDs

**Notes about Document_IDs:** 
- Although these identifiers look pretty random, there is actually a wel defined structure.
    - The first 8 characters (4 bytes) are a timestamp
    - followed by a 6 character machine identifier
    - then a 4 character process identifier
    - and finally a 6 character counter.
    
- <font color='red'> Note to consider</font>:
    - Instead of creating the default _id(s) here, we can use the _id as our given IDs in the Dataset
    - it's better to stick to the automatically created mongo IDs in order to scale well.
    - However, sometimes you badly want to prettify the never-ending ObjectID value.
        - Then, you should consider using an appropriate atomic increment strategy.
   ```javascript  
   db.coll.insert_one({_id:"myUniqueValue", a:1, b:1}) ```

In [25]:
mydb.customers.insert_one({"_id":"qwe2312312", "name": "Riccardo", "age": 35, "gender": "male"})

InsertOneResult('qwe2312312', acknowledged=True)

In [31]:
cs = customers.find({})

In [34]:
for  c in cs:
    print(c)

{'_id': ObjectId('6926c236298f27673a93246f'), 'name': 'Jane', 'age': 25, 'gender': 'female'}
{'_id': 'qwe2312312', 'name': 'Riccardo', 'age': 35, 'gender': 'male'}


## Let's filter the results

In [36]:
cs = customers.find({"age":35})
for  c in cs:
    print(c)

{'_id': 'qwe2312312', 'name': 'Riccardo', 'age': 35, 'gender': 'male'}


In [38]:
cs = customers.find({"age": {"$lt": 30}})
for  c in cs:
    print(c)

{'_id': ObjectId('6926c236298f27673a93246f'), 'name': 'Jane', 'age': 25, 'gender': 'female'}


# Selection

[operators](./mongodb-query-operators-examples.jpg)

In [39]:
cs = customers.find({"_id":"qwe2312312"})
for  c in cs:
    print(c)

{'_id': 'qwe2312312', 'name': 'Riccardo', 'age': 35, 'gender': 'male'}


# Projection

In [58]:
cs = customers.find({"age": {"$lt": 30}} # selection
                    ,{"name": 1} #projection
                   )
for  c in cs:
    print(c)

{'_id': ObjectId('6926c236298f27673a93246f'), 'name': 'Jane'}


# Sort

In [64]:
customers_to_insert = [{"name":"Caresa","age":7,"gender":"Female"},
{"name":"Bartolomeo","age":64,"gender":"Male"},
{"name":"Leslie","age":43,"gender":"Male"},
{"name":"Karmen","age":33,"gender":"Female"},
{"name":"Noble","age":98,"gender":"Male"},
{"name":"Ashby","age":86,"gender":"Male"},
{"name":"Tabbie","age":23,"gender":"Male"},
{"name":"Gareth","age":17,"gender":"Male"},
{"name":"Silvester","age":49,"gender":"Male"},
{"name":"Jolene","age":69,"gender":"Female"}]

In [65]:
mydb.customers.insert_many(customers_to_insert)

InsertManyResult([ObjectId('6926c82d298f27673a932470'), ObjectId('6926c82d298f27673a932471'), ObjectId('6926c82d298f27673a932472'), ObjectId('6926c82d298f27673a932473'), ObjectId('6926c82d298f27673a932474'), ObjectId('6926c82d298f27673a932475'), ObjectId('6926c82d298f27673a932476'), ObjectId('6926c82d298f27673a932477'), ObjectId('6926c82d298f27673a932478'), ObjectId('6926c82d298f27673a932479')], acknowledged=True)

In [99]:
two_customers= mydb.customers.find({}, {"_id":0,
                                       "name":1,
                                       "age":1}).limit(2).sort([("age",1)])
for customer in two_customers:
    print(customer)

{'name': 'Gareth', 'age': 17}
{'name': 'Tabbie', 'age': 23}


# Updates

In [75]:
customers.find_one({"name": "Bartolomeo"})

{'_id': ObjectId('6926c82d298f27673a932471'),
 'name': 'Bartolomeo',
 'age': 64,
 'gender': 'Male'}

In [79]:
mydb.customers.update_one(
    {"name": "Bartolomeo"},
    {"$set": {"age":102}}
)

UpdateResult({'n': 1, 'nModified': 1, 'ok': 1.0, 'updatedExisting': True}, acknowledged=True)

In [80]:
customers.find_one({"name": "Bartolomeo"})

{'_id': ObjectId('6926c82d298f27673a932471'),
 'name': 'Bartolomeo',
 'age': 102,
 'gender': 'Male'}

# Delete

In [85]:
customers.delete_one({"name": "Riccardo"})

DeleteResult({'n': 1, 'ok': 1.0}, acknowledged=True)

In [86]:
for c in customers.find({},{"name":1}):
    print(c)

{'_id': ObjectId('6926c236298f27673a93246f'), 'name': 'Jane'}
{'_id': ObjectId('6926c82d298f27673a932471'), 'name': 'Bartolomeo'}
{'_id': ObjectId('6926c82d298f27673a932472'), 'name': 'Leslie'}
{'_id': ObjectId('6926c82d298f27673a932473'), 'name': 'Karmen'}
{'_id': ObjectId('6926c82d298f27673a932474'), 'name': 'Noble'}
{'_id': ObjectId('6926c82d298f27673a932475'), 'name': 'Ashby'}
{'_id': ObjectId('6926c82d298f27673a932476'), 'name': 'Tabbie'}
{'_id': ObjectId('6926c82d298f27673a932477'), 'name': 'Gareth'}
{'_id': ObjectId('6926c82d298f27673a932478'), 'name': 'Silvester'}
{'_id': ObjectId('6926c82d298f27673a932479'), 'name': 'Jolene'}


# Aggregation

## Count

In [102]:
agg_result= mydb.customers.aggregate([
    {  "$group": {"_id":{"gender": "$gender"},
                  "average": {"$avg":"$age"} }}])
for gen_Avg_age in agg_result:
    print(gen_Avg_age)

{'_id': {'gender': 'Female'}, 'average': 51.0}
{'_id': {'gender': 'female'}, 'average': 25.0}
{'_id': {'gender': 'Male'}, 'average': 59.714285714285715}


## Average

In [ ]:
agg_result= mydb.customers.aggregate([
    {  "$group": {"_id":{"gender": "$gender"},
                  "average": {"$avg":"$age"} }}])
for gen_Avg_age in agg_result:
    print(gen_Avg_age)